In [ ]:
"""
=========================================================
NeuroFusionAI

Notebook : 02_image_model.ipynb

Task :
Parkinson Disease Image Classification

Model :
EfficientNet-B3 (Transfer Learning)

Author :
NeuroFusionAI

=========================================================
"""

import os
import sys

# Ensure project root is set as working directory
PROJECT_ROOT = os.path.abspath("..")
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(PROJECT_ROOT)
PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

In [ ]:
import os
import random
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision import transforms

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

warnings.filterwarnings("ignore")

In [ ]:
print("="*60)
print("GPU INFORMATION")
print("="*60)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", DEVICE)

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print("CUDA Version :", torch.version.cuda)
else:
    print("Running on CPU")

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Random Seed :", SEED)

In [ ]:
IMAGE_SIZE = 300        # EfficientNet-B3 default
BATCH_SIZE = 32
NUM_CLASSES = 2
NUM_WORKERS = 4
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
EPOCHS = 30

In [ ]:
TRAIN_DIR = "datasets/train/images"
VAL_DIR = "datasets/validation/images"
TEST_DIR = "datasets/test/images"

print(TRAIN_DIR)
print(VAL_DIR)
print(TEST_DIR)

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomAffine(
        degrees=10,
        translate=(0.05,0.05)
    ),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [ ]:
train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform
)

validation_dataset = datasets.ImageFolder(
    VAL_DIR,
    transform=test_transform
)

test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=test_transform
)

print("Train :", len(train_dataset))
print("Validation :", len(validation_dataset))
print("Test :", len(test_dataset))

In [ ]:
CLASS_NAMES = train_dataset.classes
print(CLASS_NAMES)
print(train_dataset.class_to_idx)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("DataLoaders Created Successfully")

In [ ]:
print("="*60)
print("DATASET INFORMATION")
print("="*60)
print("Training Images :", len(train_dataset))
print("Validation Images :", len(validation_dataset))
print("Testing Images :", len(test_dataset))
print()

for cls in CLASS_NAMES:
    path = os.path.join(TRAIN_DIR, cls)
    total = len(os.listdir(path))
    print(cls, ":", total)

In [ ]:
images, labels = next(iter(train_loader))

plt.figure(figsize=(12,8))
for i in range(9):
    plt.subplot(3,3,i+1)
    img = images[i].permute(1,2,0).numpy()
    img = img * np.array([0.229,0.224,0.225])
    img = img + np.array([0.485,0.456,0.406])
    img = np.clip(img,0,1)
    plt.imshow(img)
    plt.title(CLASS_NAMES[labels[i]])
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from torchvision.models import efficientnet_b3
from torchvision.models import EfficientNet_B3_Weights

In [ ]:
weights = EfficientNet_B3_Weights.DEFAULT
model = efficientnet_b3(weights=weights)
print(model)

In [ ]:
for param in model.features.parameters():
    param.requires_grad = False

print("Backbone Frozen")

In [ ]:
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features, 512),
    nn.ReLU(),
    nn.BatchNorm1d(512),
    nn.Dropout(0.3),
    nn.Linear(512, NUM_CLASSES)
)
print(model.classifier)

In [ ]:
model = model.to(DEVICE)
print("Model Loaded on", DEVICE)

In [ ]:
criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

In [ ]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

In [ ]:
if DEVICE.type == "cuda":
    scaler = torch.amp.GradScaler("cuda")
else:
    scaler = None

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    predictions = []
    labels_list = []
    
    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        optimizer.zero_grad()
        
        if DEVICE.type == "cuda" and scaler is not None:
            with torch.amp.autocast("cuda"):
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
        running_loss += loss.item()
        pred = outputs.argmax(1)
        predictions.extend(pred.cpu().numpy())
        labels_list.extend(labels.cpu().numpy())
        
    accuracy = accuracy_score(labels_list, predictions)
    return running_loss / len(loader), accuracy

In [ ]:
def validate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    predictions = []
    labels_list = []
    
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            
            if DEVICE.type == "cuda":
                with torch.amp.autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)
                
            running_loss += loss.item()
            pred = outputs.argmax(1)
            predictions.extend(pred.cpu().numpy())
            labels_list.extend(labels.cpu().numpy())
            
    accuracy = accuracy_score(labels_list, predictions)
    return running_loss / len(loader), accuracy

In [ ]:
history = {
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": []
}

In [ ]:
best_accuracy = 0
best_epoch = 0
patience = 8
counter = 0

In [ ]:
for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion
    )
    val_loss, val_acc = validate(
        model,
        validation_loader,
        criterion
    )
    scheduler.step()
    
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train Acc  : {train_acc:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Val Acc    : {val_acc:.4f}")
    
    if val_acc > best_accuracy:
        best_accuracy = val_acc
        best_epoch = epoch + 1
        counter = 0
        os.makedirs("outputs/checkpoints", exist_ok=True)
        torch.save(
            model.state_dict(),
            "outputs/checkpoints/image_best.pth"
        )
        print("✓ Best Model Saved")
    else:
        counter += 1
        
    if counter >= patience:
        print("Early Stopping")
        break

In [ ]:
print("="*60)
print("LOADING BEST MODEL")
print("="*60)

model.load_state_dict(
    torch.load(
        "outputs/checkpoints/image_best.pth",
        map_location=DEVICE
    )
)
model.eval()
print("Best model loaded successfully.")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

def test_model(model, loader):
    model.eval()
    predictions = []
    probabilities = []
    labels_list = []
    
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            pred = torch.argmax(outputs, dim=1)
            
            predictions.extend(pred.cpu().numpy())
            probabilities.extend(probs[:,1].cpu().numpy())
            labels_list.extend(labels.numpy())
            
    return labels_list, predictions, probabilities

In [ ]:
true_labels, predictions, probabilities = test_model(
    model,
    test_loader
)
print("Testing Completed")

In [ ]:
accuracy = accuracy_score(true_labels, predictions)
precision = precision_score(true_labels, predictions)
recall = recall_score(true_labels, predictions)
f1 = f1_score(true_labels, predictions)
roc_auc = roc_auc_score(true_labels, probabilities)

In [ ]:
print("="*60)
print("TEST RESULTS")
print("="*60)
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"ROC AUC   : {roc_auc:.4f}")

In [ ]:
print("="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(
    classification_report(
        true_labels,
        predictions,
        target_names=CLASS_NAMES
    )
)

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(true_labels, predictions)
print(cm)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xticks([0,1],CLASS_NAMES)
plt.yticks([0,1],CLASS_NAMES)

for i in range(2):
    for j in range(2):
        plt.text(
            j,
            i,
            cm[i,j],
            ha="center",
            va="center",
            fontsize=14
        )
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
from sklearn.metrics import roc_curve
fpr,tpr,_ = roc_curve(true_labels, probabilities)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, linewidth=2, label=f"AUC = {roc_auc:.3f}")
plt.plot([0,1],[0,1],'--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve
precision_curve, recall_curve, _ = precision_recall_curve(true_labels, probabilities)

plt.figure(figsize=(6,6))
plt.plot(recall_curve, precision_curve, linewidth=2)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision Recall Curve")
plt.grid(True)
plt.show()

In [ ]:
metrics = {
    "Accuracy":accuracy,
    "Precision":precision,
    "Recall":recall,
    "F1":f1,
    "ROC_AUC":roc_auc
}
os.makedirs("outputs/reports", exist_ok=True)
pd.DataFrame([metrics]).to_csv(
    "outputs/reports/image_metrics.csv",
    index=False
)
print("Metrics Saved")

In [ ]:
report = classification_report(
    true_labels,
    predictions,
    target_names=CLASS_NAMES
)
os.makedirs("outputs/reports", exist_ok=True)
with open("outputs/reports/image_classification_report.txt", "w") as f:
    f.write(report)
print("Classification Report Saved")

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history["train_acc"], label="Train Accuracy")
plt.plot(history["val_acc"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training Accuracy")
plt.legend()
plt.grid(True)
os.makedirs("outputs/plots", exist_ok=True)
plt.savefig("outputs/plots/image_accuracy.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.legend()
plt.grid(True)
os.makedirs("outputs/plots", exist_ok=True)
plt.savefig("outputs/plots/image_loss.png", dpi=300)
plt.show()

In [ ]:
history_df = pd.DataFrame(history)
os.makedirs("outputs/logs", exist_ok=True)
history_df.to_csv(
    "outputs/logs/image_training_history.csv",
    index=False
)
print("Training history saved.")

In [ ]:
os.makedirs("outputs/checkpoints", exist_ok=True)
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "history": history,
        "best_accuracy": best_accuracy,
        "best_epoch": best_epoch
    },
    "outputs/checkpoints/image_complete_checkpoint.pth"
)
print("Complete checkpoint saved.")

In [ ]:
misclassified = []
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(1)
        
        for img, pred, label in zip(images, preds, labels):
            if pred.cpu() != label:
                misclassified.append(
                    (
                        img.cpu(),
                        label.item(),
                        pred.cpu().item()
                    )
                )
print("Misclassified Images :", len(misclassified))

In [ ]:
plt.figure(figsize=(12,8))
for i in range(min(9, len(misclassified))):
    img, true_label, pred_label = misclassified[i]
    img = img.permute(1,2,0).numpy()
    img = img * np.array([0.229,0.224,0.225])
    img = img + np.array([0.485,0.456,0.406])
    img = np.clip(img,0,1)
    
    plt.subplot(3,3,i+1)
    plt.imshow(img)
    plt.title(f"T:{CLASS_NAMES[true_label]}\nP:{CLASS_NAMES[pred_label]}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
def predict_image(image_path):
    image = Image.open(image_path).convert("RGB")
    image = test_transform(image)
    image = image.unsqueeze(0)
    image = image.to(DEVICE)
    
    model.eval()
    with torch.no_grad():
        output = model(image)
        probability = torch.softmax(output, dim=1)
        prediction = output.argmax(1).item()
        
    return CLASS_NAMES[prediction], probability[0][prediction].item()

In [ ]:
test_parkinson_dir = "datasets/test/images/parkinson"
sample = "datasets/test/images/parkinson/example.png"
if not os.path.exists(sample) and os.path.exists(test_parkinson_dir):
    files = [f for f in os.listdir(test_parkinson_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    if files:
        sample = os.path.join(test_parkinson_dir, files[0])

if os.path.exists(sample):
    label, confidence = predict_image(sample)
    print("Prediction :", label)
    print("Confidence :", round(confidence * 100, 2), "%")
else:
    print("Replace sample path with a test image.")

In [ ]:
print("="*60)
print("IMAGE MODEL TRAINING SUMMARY")
print("="*60)
print(f"Best Epoch        : {best_epoch}")
print(f"Best Accuracy     : {best_accuracy:.4f}")
print(f"Training Images   : {len(train_dataset)}")
print(f"Validation Images : {len(validation_dataset)}")
print(f"Testing Images    : {len(test_dataset)}")
print("\nSaved Files")
print("-"*40)
print("✓ outputs/checkpoints/image_best.pth")
print("✓ outputs/checkpoints/image_complete_checkpoint.pth")
print("✓ outputs/logs/image_training_history.csv")
print("✓ outputs/reports/image_metrics.csv")
print("✓ outputs/reports/image_classification_report.txt")
print("✓ outputs/plots/image_accuracy.png")
print("✓ outputs/plots/image_loss.png")
print("="*60)
print("IMAGE MODEL COMPLETED SUCCESSFULLY")
print("="*60)